# Text to image with classifier-free guidance

[Notebook 02](02-train-a-diffusion-model.ipynb) trained a model that knows what its images look like but cannot be told which one to draw. This notebook conditions the same kind of DiT on text. An encoder turns each caption into a `TextContext`, the objective hands it to the model under the `textcontext` keyword, and it drops the caption on a fraction of rows (`unconditional_prob`) so one model learns both the conditional and the unconditional distribution. At sampling time classifier-free guidance extrapolates from the unconditional prediction toward the captioned one, and `CFG(scale)` says how far.

Two routes share every line but the configuration cell:

- **Offline route (default, runs here).** Generated stripe images captioned `"horizontal"` or `"vertical"`, encoded by `CharTable`, a fixed random character table with the shape of a real text encoder. It runs on a CPU in a few minutes and shows the conditioning and guidance mechanics with captions the model can actually learn.
- **Oxford Flowers route.** `ROUTE = "flowers"` uses the prepared dataset from notebook 02, the CLIP-L/14 text tower and, when `USE_VAE` is set, the Stable Diffusion VAE so the model denoises 8x-downsampled latents. The first run downloads CLIP and the VAE (about 2 GB); training at 128 pixels wants a GPU or TPU and roughly an hour. That route was not executed while writing this notebook.

In [ ]:
# On Colab: install dew and the JAX build for the runtime. Locally this cell is a no-op.
try:
    import google.colab  # noqa: F401
    import subprocess, sys
    try:
        import jax
        tpu = any("tpu" in str(d).lower() for d in jax.devices())
    except Exception:
        tpu = False
    extra = "jax[tpu]" if tpu else "jax[cuda12]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew", extra])
except ImportError:
    pass

In [ ]:
ROUTE = "offline"          # "offline" runs anywhere; "flowers" needs the prepared dataset, CLIP and an accelerator
RUN_DIR = "runs/03-text-to-image"

if ROUTE == "offline":
    IMAGE_SIZE = 16
    BATCH_SIZE = 16
    STEPS = 300
    MODEL = dict(patch_size=4, emb_features=48, num_layers=2, num_heads=2, mlp_ratio=2)
    DTYPE, ATTENTION = "float32", "xla"
    USE_VAE = False
    PROMPTS = ("horizontal", "vertical", "horizontal", "vertical")
    SAMPLE_STEPS = 8
else:
    FLOWERS_PATH = "~/dew-data/tfds-arrayrecord/oxford_flowers102/2.1.1"  # builder.data_dir from preparation
    IMAGE_SIZE = 128
    BATCH_SIZE = 32
    STEPS = 15_000
    MODEL = dict(patch_size=2, emb_features=384, num_layers=8, num_heads=6)
    DTYPE, ATTENTION = "bfloat16", "auto"
    USE_VAE = True
    PROMPTS = ("a water lily", "a sunflower", "a red rose", "a purple orchid")
    SAMPLE_STEPS = 40
GUIDANCE_SCALES = (1.0, 2.0, 4.0)  # one row of the final grid each
SEED = 0

In [ ]:
import jax

print(jax.devices())
print(jax.default_backend())

## Captioned data

A captioning dataset carries its text as strings, and `load(tokenize=inputs.tokenize)` is where the run's own condition turns them into token arrays before the batch reaches a device. That is why the `InputSpec` is built first: the encoder it names decides the ids and the context length, and the dataset only carries words.

The offline route builds stripe images whose caption is their orientation, and tokenizes them the same way through the spec.

In [ ]:
import numpy as np
from dew import Condition, Field, InputSpec
from dew.data import Dataset, Loading

if ROUTE == "offline":
    from dew.inputs import CharTable
    encoder = CharTable.from_pretrained(tokens=12, features=16)
else:
    from dew.inputs import CLIPText
    encoder = CLIPText.from_pretrained("openai/clip-vit-large-patch14")

inputs = InputSpec(
    sample=Field("image", (IMAGE_SIZE, IMAGE_SIZE, 3)),
    conditions={"textcontext": Condition(encoder, field="text", unconditional="")},
)


def stripe_images(count, size, seed):
    rng = np.random.default_rng(seed)
    images = np.zeros((count, size, size, 3), np.uint8)
    labels = rng.integers(0, 2, count)
    for i in range(count):
        colour = rng.integers(64, 256, 3)
        period = int(rng.integers(2, 5))
        rows = (np.arange(size) // period) % 2 == 0
        # label 0: bands across the image vary with the row index; label 1: with the column
        pattern = rows[:, None] if labels[i] == 0 else rows[None, :]
        images[i][np.broadcast_to(pattern, (size, size))] = colour
    return images, labels.astype(np.int32)


if ROUTE == "offline":
    CAPTIONS = np.array(["horizontal", "vertical"])
    train_images, train_labels = stripe_images(512, IMAGE_SIZE, seed=SEED)
    val_images, val_labels = stripe_images(BATCH_SIZE, IMAGE_SIZE, seed=SEED + 1)

    class StripeBatches:
        def __init__(self):
            self.index = 0

        def __iter__(self):
            return self

        def __next__(self):
            rows = np.random.default_rng(SEED + self.index).choice(
                len(train_images), BATCH_SIZE, replace=False)
            self.index += 1
            return {"image": train_images[rows],
                    **inputs.tokenize(list(CAPTIONS[train_labels[rows]]))}

        def get_state(self):
            return str(self.index).encode()

        def set_state(self, state):
            self.index = int(state.decode())

    val_batch = {"image": val_images, **inputs.tokenize(list(CAPTIONS[val_labels]))}
    data = Dataset(train=StripeBatches, val=lambda: iter([val_batch]),
                   records=len(train_images), batch=BATCH_SIZE)
else:
    from dew.data import OxfordFlowers
    data = OxfordFlowers(path=FLOWERS_PATH, image_size=IMAGE_SIZE, val_batches=2,
                         loading=Loading(workers=4, threads=4, read_buffer=16, worker_buffer=2)
                         ).load(batch=BATCH_SIZE, tokenize=inputs.tokenize)

batch = next(iter(data.val()))
print({key: (value.shape, value.dtype) for key, value in batch["text"].items()})
print("captions:", encoder.captions(batch["text"])[:4])

## The model, the process and the objective

The model is the same DiT as notebook 02 with a text stream: `SimpleDiT` takes the `textcontext` keyword the spec names. The process is the EDM preset again. With `USE_VAE`, a `StableDiffusionVAE` sits in front of the objective: `encode` compresses each image to latents before the loss and `decode` expands sampled latents back to pixels, and `output_channels` follows the latent channel count.

`DiffusionObjective` freezes the encoder and the autoencoder as state alongside the model's parameters, so they ride in checkpoints without being trained.

In [ ]:
import optax
from dew import Checkpoints, Trainer, models, presets
from dew.objectives.diffusion import DiffusionObjective
from dew.sampling import CFG, EulerAncestral

autoencoder = None
if USE_VAE:
    from dew.nn.autoencoders.sd_vae import StableDiffusionVAE
    autoencoder = StableDiffusionVAE()

channels = 3 if autoencoder is None else autoencoder.latent_channels
process = presets.EDM()()
model = models.build("simple_dit", **MODEL, output_channels=channels,
                     dtype=DTYPE, attention_impl=ATTENTION)
objective = DiffusionObjective(model, process, inputs, autoencoder=autoencoder,
                               unconditional_prob=0.12, ema_decay=0.99,
                               sampler=EulerAncestral(), guidance=CFG(2.0), steps=SAMPLE_STEPS)
trainer = Trainer(objective, optax.adamw(1e-3 if ROUTE == "offline" else 2e-4),
                  key=jax.random.key(SEED), checkpoints=Checkpoints(RUN_DIR))
state = trainer.fit(data, steps=STEPS, log_every=max(1, STEPS // 5), checkpoint_every=STEPS)
print("optimizer updates:", int(state.updates))

## Classifier-free guidance

At sampling time the denoiser sees two conditions: the caption and the empty one. `process.denoiser(model, params, conditions, unconditional)` bundles both, and `CFG(scale)` turns the pair into `uncond + scale * (cond - uncond)` at every step. Scale 1 is the plain conditional prediction; larger scales follow the caption harder and trade away variety. `CFG(scale, interval=(0.1, 0.9))` limits the extrapolation to the middle of the trajectory, where structure forms.

`objective.trainable(params)` strips the frozen encoder and autoencoder collections out of the variables tree, leaving the model's own; `objective.encode` runs each condition's encoder over its tokens, keyed by the model keyword (`textcontext`), and `objective.unconditional` is the empty caption already encoded.

In [ ]:
from PIL import Image

try:
    from IPython.display import display
except ModuleNotFoundError:  # running the cells as a plain script
    def display(image):
        print(f"image {image.width}x{image.height}")

from dew.sampling import sample


def show_grid(frames, cols, scale=4):
    frames = np.asarray(frames)
    rows = (len(frames) + cols - 1) // cols
    h, w, c = frames.shape[1:]
    grid = frames.reshape(rows, cols, h, w, c).transpose(0, 2, 1, 3, 4).reshape(rows * h, cols * w, c)
    image = Image.fromarray(grid)
    display(image.resize((image.width * scale, image.height * scale), Image.NEAREST))
    return image


def generate(params, prompts, guidance, key):
    given = objective.encode(params["encoders"],
                             {"textcontext": encoder.tokenize(list(prompts))})
    denoise = process.denoiser(model, objective.trainable(params), given,
                               unconditional=objective.unconditional)
    x_T = process.noise(key, (len(prompts), *objective.latent_shape))
    images = sample(denoise, x_T, SAMPLE_STEPS, solver=EulerAncestral(),
                    guidance=guidance, key=key)
    if autoencoder is not None:
        images = autoencoder.decode(params["autoencoder"], images)
    return np.clip(np.asarray(images), -1, 1)


rows = []
for scale in GUIDANCE_SCALES:
    guidance = None if scale == 1.0 else CFG(scale, interval=(0.1, 0.9))
    rows.append(generate(state.averaged, PROMPTS, guidance, jax.random.key(1)))
    print(f"guidance scale {scale}: done")

frames = np.clip((np.concatenate(rows) + 1) * 127.5, 0, 255).astype(np.uint8)
import os
os.makedirs(RUN_DIR, exist_ok=True)
show_grid(frames, cols=len(PROMPTS)).save(f"{RUN_DIR}/guidance-grid.png")

## Read the grid

Each row is one guidance scale, each column one prompt. On the offline route the model has two captions to learn, so check that the `"horizontal"` columns come out as horizontal stripes and the `"vertical"` ones as vertical; the measurement below does that numerically, by comparing the variance of each sample along rows against along columns. On the flowers route the columns follow their captions more closely as the scale grows, and past about 4 the colours saturate and the four columns drift toward the same look.

In [ ]:
def orientation(image):
    """0 for horizontal bands (pixels vary down the rows), 1 for vertical (across the columns)."""
    grey = image.mean(axis=-1)
    return int(grey.var(axis=1).mean() > grey.var(axis=0).mean())


if ROUTE == "offline":
    wanted = [0 if prompt == "horizontal" else 1 for prompt in PROMPTS]
    for scale, row in zip(GUIDANCE_SCALES, rows):
        got = [orientation(image) for image in row]
        print(f"scale {scale}: wanted {wanted} got {got}")

## Where to go next

`recipes/diffusion/train.py` runs this configuration from the command line, with `text:` and `autoencoder` flags for the encoder and the VAE, and writes `run.json` beside the checkpoints so `TextToImage.from_run(directory)` rebuilds the pipeline for sampling. [Notebook 04](04-samplers-and-schedules.ipynb) compares the solvers on a trained checkpoint.